In [2]:
# Installations

!pip install -q transformers datasets peft accelerate bitsandbytes sentence-transformers faiss-cpu evaluate

In [3]:
# Setup

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [4]:
# LOAD REAL DATASET (SQuAD)

from datasets import load_dataset

dataset = load_dataset("squad")

def preprocess(example):
    q = example["question"]
    if "what" in q.lower():
        label = 0
    elif "solve" in q.lower():
        label = 1
    else:
        label = 2
    return {"text": q, "label": label}

dataset = dataset["train"].select(range(5000)).map(preprocess)
dataset = dataset.train_test_split(test_size=0.1)

In [5]:
# TOKENIZATION

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(x):
    return tokenizer(x["text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [28]:
# LoRA TRAINING

from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin","v_lin"],
    lora_dropout=0.1
)

model = get_peft_model(model, lora_config)
model.to(device)
model.eval() # Added

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModel(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
                  (lora_B): Mod

In [29]:
# TRAIN + F1

import evaluate, numpy as np
from transformers import Trainer, TrainingArguments

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return f1_metric.compute(predictions=preds, references=labels, average="weighted")

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    fp16=True,
    # evaluation_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss
500,0.691112


TrainOutput(global_step=846, training_loss=0.513316341607565, metrics={'train_runtime': 158.8118, 'train_samples_per_second': 85.006, 'train_steps_per_second': 5.327, 'total_flos': 1794457069056000.0, 'train_loss': 0.513316341607565, 'epoch': 3.0})

In [30]:
# SAVE MODEL

model.save_pretrained("cls_model")
tokenizer.save_pretrained("cls_model")

('cls_model/tokenizer_config.json', 'cls_model/tokenizer.json')

In [39]:
# FREE MEMORY

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [1]:
# GENERATOR

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

# Quantization config (FIXED)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True   # IMPORTANT
)

tokenizer_mistral = AutoTokenizer.from_pretrained(model_name)

# Fix warning
tokenizer_mistral.pad_token = tokenizer_mistral.eos_token

model_mistral = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [42]:
# RAG (FAISS)

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. LOAD DATASET (SciQ)
rag_data = load_dataset("sciq")

# Build proper documents (support + answer)
def build_docs(example):
    return {
        "text": example["support"] + " " + example["correct_answer"]
    }

docs_data = rag_data["train"].select(range(3000)).map(build_docs)
docs = docs_data["text"]


# 2. CHUNKING
def chunk_text(text, chunk_size=100):
    words = text.split()
    return [
        " ".join(words[i:i+chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

cleaned_docs = []

for doc in docs:
    chunks = chunk_text(doc)

    for chunk in chunks:
        # remove bad chunks
        if len(set(chunk.split())) < 5:   # repetitive
            continue
        if len(chunk.split()) < 20:       # too short
            continue

        cleaned_docs.append(chunk)

chunked_docs = cleaned_docs


# 3. EMBEDDINGS + FAISS
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(
    chunked_docs,
    show_progress_bar=True
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))


# 4. RETRIEVAL FUNCTION
def retrieve_context(query, k=5, max_words=120):
    q_vec = embed_model.encode([query])
    _, I = index.search(np.array(q_vec), k)

    results = []

    for i in I[0]:
        chunk = chunked_docs[i]

        # skip bad chunks again
        if len(set(chunk.split())) < 5:
            continue

        results.append(chunk)

    # fallback if nothing good found
    if not results:
        return "Sorry! Answer Unavailable."

    context = " ".join(results)
    return " ".join(context.split()[:max_words])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/93 [00:00<?, ?it/s]

Some New Features added!

In [43]:
# ================= NEW FEATURES =================

# YouTube Search
!pip install -q youtube-search-python

from youtubesearchpython import VideosSearch

def get_youtube_links(query, max_results=2):
    try:
        videos_search = VideosSearch(query, limit=max_results)
        results = videos_search.result()["result"]

        links = []
        for v in results:
            links.append(f"{v['title']}: {v['link']}")

        return "\n".join(links)
    except:
        return "No video found."


# ================= MCQ GENERATOR =================
def generate_mcq(query, level):
    try:
        difficulty = {
            "beginner": "easy",
            "intermediate": "moderate",
            "advanced": "hard"
        }.get(level, "easy")

        mcq_prompt = f"""
<s>[INST]
Create 1 {difficulty} MCQ based on:

{query}

Format:
Question:
A)
B)
C)
D)
Answer:
[/INST]
"""

        inputs = tokenizer_mistral(mcq_prompt, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model_mistral.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

        decoded = tokenizer_mistral.decode(outputs[0], skip_special_tokens=True)
        return decoded.split("[/INST]")[-1].strip()

    except Exception as e:
        print("MCQ ERROR:", e)
        return "MCQ generation failed."


# ================= MCQ CHECKER (WITH SCORE) =================
def check_mcq_answer(user, mcq, user_answer):
    correct = None

    for line in mcq.split("\n"):
        if "Answer:" in line:
            correct = line.split("Answer:")[-1].strip().upper()

    if correct is None:
        return "Couldn't evaluate."

    # Initialize user if not exists
    if user not in student_db:
        student_db[user] = {
            "level": "beginner",
            "history": [],
            "score": 0,
            "attempts": 0
        }

    student_db[user]["attempts"] += 1

    if user_answer.strip().upper() == correct:
        student_db[user]["score"] += 1
        result = f"Correct! ({correct})"
    else:
        result = f"rong. Correct answer is {correct}"

    score = student_db[user]["score"]
    attempts = student_db[user]["attempts"]

    return f"{result}\n\nScore: {score}/{attempts}"


# ================= VOICE =================
!pip install -q gtts SpeechRecognition

import speech_recognition as sr
from gtts import gTTS

def speech_to_text(audio_file):
    r = sr.Recognizer()
    with sr.AudioFile(audio_file) as source:
        audio = r.record(source)

    try:
        return r.recognize_google(audio)
    except:
        return "Could not understand audio"


def text_to_speech(text):
    tts = gTTS(text)
    file = "response.mp3"
    tts.save(file)
    return file


# ================= SCORE FETCH =================
def get_score(user):
    if user not in student_db:
        return "0/0"

    s = student_db[user]["score"]
    a = student_db[user]["attempts"]

    return f"{s}/{a}"

In [12]:
# Student Memory

student_db = {}

def update_student(user, q, cat):
    if user not in student_db:
        student_db[user] = {"level": "beginner", "history": []}

    student_db[user]["history"].append(q)

    if len(student_db[user]["history"]) > 10:
        student_db[user]["level"] = "intermediate"
    if len(student_db[user]["history"]) > 25:
        student_db[user]["level"] = "advanced"

def get_level(user):
    return student_db.get(user, {}).get("level", "beginner")

In [44]:
# Prompt + Generation

def build_prompt(query, level, context):
    return f"""
<s>[INST]
You are a helpful AI tutor.

Explain this concept for a {level} student.

Question: {query}

Give:
- A simple definition
- Step-by-step explanation (4 steps)

Do not copy instructions. Explain clearly.

[/INST]
"""

def generate_answer(prompt):
    try:
        inputs = tokenizer_mistral(prompt, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model_mistral.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

        decoded = tokenizer_mistral.decode(outputs[0], skip_special_tokens=True)
        answer = decoded.split("[/INST]")[-1].strip()

        if not answer.endswith((".", "!", "?")):
            answer = answer.rsplit(".", 1)[0] + "."

        return answer

    except Exception as e:
        print("GEN ERROR:", e)
        return "Error generating answer."

In [45]:
# Classification + Pipeline

label_map = {0: "Concept", 1: "Problem", 2: "Theory"}

def classify(q):
    try:
        inputs = tokenizer(q, return_tensors="pt", truncation=True, padding=True)

        inputs = {k: v.to(device) for k, v in inputs.items()}  # FIX

        with torch.no_grad():   # IMPORTANT
            outputs = model(**inputs)

        return label_map[torch.argmax(outputs.logits).item()]

    except Exception as e:
        print("CLASSIFY ERROR:", e)
        return "Not Available"   # fallback


def tutor_response(query, user="student"):
    category = classify(query)

    update_student(user, query, category)
    level = get_level(user)

    context = retrieve_context(query)

    # keep your existing prompt
    prompt = build_prompt(query, level, context)
    answer = generate_answer(prompt)

    # ADD THESE 2 LINES ONLY
    videos = get_youtube_links(query + " explanation for students")
    mcq = generate_mcq(query, level)

    # MODIFY RETURN
    return category, answer, level, videos, mcq

In [47]:
# Testing the System

tutor_response("Find speed if distance = 100m and time = 10s","student1")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


('Theory',
 'Simple Definition:\n\nSpeed is a measure of how fast an object moves or travels. It is calculated by dividing the distance traveled by the time it took to travel that distance.\n\nStep-by-Step Explanation:\n\n1. We are given the distance traveled as 100 meters and the time taken as 10 seconds.\n2. To find the speed, we need to divide the distance by the time.\n3. So, the speed is 100 meters divided by 10 seconds.\n4. Therefore, the speed is 10 meters per second (m/s).\n\nIn summary, speed is calculated by dividing the distance traveled by the time it took to travel that distance. In this case, the distance is 100 meters and the time is 10 seconds, so the speed is 10 meters per second.',
 'intermediate',
 'No video found.',
 'Question:\nA) 10 m/s\nB) 10 km/h\nC) 100 km/h\nD) 1000 km/h\n\nAnswer: A) 10 m/s')

In [25]:
# Gradio UI

!pip install -q gradio

In [40]:
# UI (NEED TO BE FIXED)

import gradio as gr

# Chat function
def chat(user, query):
    try:
        if not query:
            return "", "", "", "", "", None, ""

        category, answer, level, videos, mcq = tutor_response(query, user)

        audio = text_to_speech(answer)
        score = get_score(user)

        return category, level, answer, videos, mcq, audio, score

    except Exception as e:
        print("ERROR:", str(e))   # this will show real error in colab
        return "Error", "Error", "Error", "Error", "Error", None, "Error"


# MCQ Evaluation
def evaluate_answer(user, mcq, user_ans):
    return check_mcq_answer(user, mcq, user_ans)


# Feedback function
def save_feedback(query, answer, feedback):
    with open("feedback.txt", "a") as f:
        f.write(f"{query}|{answer}|{feedback}\n")
    return "Feedback saved!"


with gr.Blocks() as demo:
    gr.Markdown("### AI Tutor")

    # Inputs
    user = gr.Textbox(label="Enter your name", value="student1")
    query = gr.Textbox(label="Ask a question")

    ask_btn = gr.Button("Ask")

    # Outputs
    category = gr.Textbox(label="Category")
    level = gr.Textbox(label="Level")
    answer = gr.Textbox(label="Answer")

    videos = gr.Textbox(label="YouTube Videos")
    mcq = gr.Textbox(label="MCQ")

    audio_output = gr.Audio(label="AI Voice Response")
    score_display = gr.Textbox(label="Score")

    # MATCHES EXACT RETURN (7 outputs)
    ask_btn.click(
        chat,
        inputs=[user, query],
        outputs=[category, level, answer, videos, mcq, audio_output, score_display]
    )

    # ===== MCQ SECTION =====
    gr.Markdown("### Answer the MCQ")

    user_ans = gr.Textbox(label="Your Answer (A/B/C/D)")
    result = gr.Textbox(label="Result")

    check_btn = gr.Button("Check Answer")

    check_btn.click(
        evaluate_answer,
        inputs=[user, mcq, user_ans],
        outputs=result
    )

    # ===== FEEDBACK =====
    gr.Markdown("### Feedback")

    feedback = gr.Radio(["Yes", "No"], label="Was this helpful?")
    feedback_btn = gr.Button("Submit Feedback")
    feedback_status = gr.Textbox(label="Status")

    feedback_btn.click(
        save_feedback,
        inputs=[query, answer, feedback],
        outputs=feedback_status
    )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://98cf5ea0aba49645bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [41]:
# UI 2 (NEED TO BE FIXED)

import gradio as gr

# Chat function
def chat_interface(user, message, history):
    try:
        if not message:
            return history, ""

        category, answer, level, videos, mcq = tutor_response(message, user)

        score = get_score(user)

        full_response = f"""
 Category: {category}
 Level: {level}

 Answer:
{answer}

 Videos:
{videos}

 MCQ:
{mcq}

 Score: {score}
"""

        history.append((message, full_response))
        return history, ""

    except Exception as e:
        print("CHAT ERROR:", e)
        history.append((message, " Error occurred"))
        return history, ""

# MCQ evaluation
def check_answer(user, mcq, ans):
    return check_mcq_answer(user, mcq, ans)


with gr.Blocks() as demo:
    gr.Markdown("## AI Tutor")

    user = gr.Textbox(label="Enter your name", value="student1")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(label="Ask your question")
    send_btn = gr.Button("Send")

    # MCQ answer section
    gr.Markdown("### Answer MCQ")

    mcq_box = gr.Textbox(label="Paste MCQ here (auto shown above)")
    ans_box = gr.Textbox(label="Your Answer (A/B/C/D)")
    result_box = gr.Textbox(label="Result")

    check_btn = gr.Button("Check Answer")

    # Button actions
    send_btn.click(
        chat_interface,
        inputs=[user, msg, chatbot],
        outputs=[chatbot, msg]
    )

    check_btn.click(
        check_answer,
        inputs=[user, mcq_box, ans_box],
        outputs=result_box
    )

demo.launch()

/tmp/ipykernel_8386/3776770532.py:49: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_8386/3776770532.py:49: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d21d48f44af91257d2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Q1-** How does electricity flow in a circuit?

**Ans-** (As given by the AI Tutor) Simple Definition: Electricity is a form of energy that flows through a circuit, which is a closed loop of conductive material. The flow of electricity is caused by the movement of charged particles, called electrons, from one atom to another.

Step-by-Step Explanation:

1. A power source, such as a battery or an outlet, provides an electrical potential difference, or voltage, which creates an electric field within the circuit. This electric field is a force that drives the electrons to move in a particular direction.
2. The electrons flow through the circuit, following the path of least resistance. This means that they will take the path that allows them to move the easiest and fastest, which is often through a wire with a low resistance.
3. As the electrons flow through the circuit, they interact with other charged particles, such as atoms and other electrons. This interaction creates a current, which is the flow of electrons through the circuit.
4. The current flows through the circuit until it reaches a load, which is a device or component that uses the electricity to perform a function. The load can be a light bulb, a motor, or any other device that requires electricity to operate. Once the current reaches the load, it is converted into another form of energy, such as light or motion, and the circuit is complete.

**Q2-** Find speed if distance = 100m and time = 10s

**Ans-** (By AI Tutor) Simple Definition: Speed is a measure of how fast an object moves or travels. It is calculated by dividing the distance traveled by the time it took to travel that distance.

Step-by-Step Explanation:

1.	First, we need to understand the given information. We are asked to find the speed of an object that traveled a distance of 100 meters in 10 seconds.
2.	Next, we can use the formula for speed, which is:
3.	Speed = Distance / Time
4.	Plugging in the given values, we get:
5.	Speed = 100 meters / 10 seconds
6.	Solving for speed, we get:
7.	Speed = 10 meters/second
So, the speed of the object is 10 meters per second.

**Q3-** Solve 2x + 5 = 15

**Ans-** (By AI Tutor) Simple Definition: Solving an equation is finding the value of the variable (in this case, x) that makes the equation true.

Step-by-Step Explanation:

1.	Start by isolating the variable (x) on one side of the equation. To do this, subtract 5 from both sides of the equation: 2x + 5 - 5 = 15 - 5.
2.	Simplify the equation: 2x = 10.
3.	Divide both sides of the equation by 2 to find the value of x: 2x / 2 = 10 / 2.
4.	Simplify the equation: x = 5.
So, the solution to the equation 2x + 5 = 15 is x = 5
